# One Loop of Training

In this notebook, a very simple neural network (one layer with one neuron) is trained for one loop (one epoch) with simple inputs (one sample with one item and one output). Idea is to show how the parameters (weight and bias) get original random values and then gradients are calculated and applied to the parameters and nudge the neural network toward accurate prediction.

## 1. Imports

In the neural network training with PyTorch, there are 3 primary modules/submodules that are imported in every deep learning project. There are many more, but these three are primary:

- torch
- torch.nn
- torch.optim

### 1.1 torch

`torch` is the root package for PyTorch. It has lot of functionality, including tensors (Tensor class and tensor factory), autograd (functionality to automatically calculate gradients). This root module `torch` contains many submodules, include nn, optim, utils, etc.

### 1.2 torch.nn

`torch.nn` submodule contains the classes to define the architecture of a neural network. For example, a Linear layer with appropriate inputs and outputs are defined with `nn.Linear` class. A sequential container for the layers done with `nn.Sequential`. Creating a custom set of layers with a custom forward flow through the layers is done with `nn.Module`.

### 1.3 torch.optim

Optimizer classes define functionality for adjusting the parameters (so that the loss steadily gets lowered) based on the calculated gradients. SGD (Stochastic Gradient Descent) is a popular optimizer algorithm where the weights are lowered by the gradient multiplied by the learning rate. There are many other optimizer algorithms like Adam that consider the current and previous gradients to adjust the parameters so that loss is reduced and inferences are more accurate.


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim

## 2. A seed for PyTorch

PyTorch generates random values for the parameters (weights and biases) when a neural network is created. The training loop (over multiple iterations/epochs) adjusts these initial values into something that produces more accurate inference. Assigning a seed lets PyTorch create the same random values between training sessions. This way, you can see how the parameters are being changed between the training runs. Otherwise, if the random values are different each time, they would change differently each time, and comparing/observing will not result in meaningful insights.

This seed is called manual_seed. Any number can be used, but you'd see 42 being used in several places (the answer for everything is 42, in the popular Douglas Adams novel)

In [27]:
torch.manual_seed(42)

## 3. Model Architecture

For a neural network model, we need the following 3 things:

- Model Layers / Architecture
- Loss Function
- Optimizer

### 3.1 Model Layers / Architecture

For the example, the neural network is very simple. It is a Linear layer with one neuron. It takes one input and produces one output. The calculation that happens at a neuron is: y = wx + b. Here, w (weight) and b (bias) are called parameters. And y is the output. There will always be one output per neuron. If the layer has 64 neurons, then that layer has 64 outputs (one each from a neuron). The outputs of a layer will become inputs to the following layer.

In this example, there is only one layer with one input and one output, which comes to `nn.Linear(1, 1)`. The input will be the one sample with one item tensor we are going to define. And the output will be the inference of entire neural network.

### 3.2 Loss Function

We need create function that calculates the loss from one run (one epoch or batch from a large number of samples) using a certain algorithm. Here we use the popular loss function MSE - Mean Squared Error. Final inference minus the actual label (the correct output for a given set of inputs) is the Error. Square it, so that negatives and positives don't cancel each other out. Then calculate a mean across the samples. That is MSE loss. There are many other loss algorithms as well, e.g. Cross Entropy Loss, which is used in classification systems (e.g. recognizing digits from 0 through 9).

`loss.backward()` calculates the gradients, which can be interpreted as how much each parameter (w1, w2, ... b1, b2, etc.) at each neuron is responsible for the eventual loss.

### 3.3 Optimizer

Once the gradients are calculated, they need to be used in adjusting the parameters. Parameters (w, b) need to be adjusted (increased or decreased) so that the next y_prediction for the neural network (y = wx + b at each neuron) need to be closer to the y_actual (the real output value from the data). A popular Optimizer, SGD, is used here. This takes the gradients calculated above and multiplies with a learning rate (something like 0.01) and removes from the parameter (if the gradient is positive, the parameter value will go down; and if it is negative, the parameter value will go up).

In [28]:
# One linear layer with one input and one output (i.e. one neuron)
model = nn.Linear(1, 1)

# Use Mean Squared Error to calculate loss
loss_function = nn.MSELoss()

# Stochastic Gradient Descent is used as optimizer
# Optimizer adjusts model parameters with the help of gradients and learning rate 
optimizer = optim.SGD(model.parameters(), lr=0.1)

## 4. Print Parameter Info

Following is a print function that prints the values of a parameter (w, b), whether requires_grad is set to true, and what is current gradient. For parameters (w, b), the requires_grad is set to true by PyTorch; it is not set to true for inputs and outputs like x, y (the samples and labels). When autograd is set to true, the calculated gradients are applied. We would want parameters (w, b) changing and eventually come to an optimal value. This is not the case for inputs and outputs (x, y in the data) - they remain the same throughout; they don't change.

In [29]:
def print_parameter_info (model, location_info):
    """
    Prints information about the parameters in a model. This can be called at various locations in a training run.

    parameters:
    model: the neural network model
    location_info: a string describing where this function is called from

    return:
    None. Info is printed.
    """

    print(f"\n{location_info}:")
    for param in model.parameters():
        print(f"param: {param.data}, requires_grad: {param.requires_grad}, gradient: {param.grad}")

## 5. Parameters at the beginning

When a model is created (here with a simple `Linear(1,1)`, the parameters are initialized with random values. Here we have one layer with one neuron, so, there will be one weight (w) and one bias (b). This is the tiniest of the models with 2 parameters.

```
Right after initialization:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None
```

Since the seed is set at the beginning (`torch.manual_seed(42)`), the parameter tensors will have the same values during a repeated invocation of this code.

In [30]:
print_parameter_info(model, "Right after initialization")


Right after initialization:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None


## 6. Data (inputs and outputs)

To train any neural network, we need data. This is a super simple neural network create to see how parameters are changing through the lifecycle of training. To make it simple, there will be one input (x = 2) and one output (y = 3). So, this neural network needs to figure out what are the parameters (w, b) in the equation: y = wx + b. Obviously, for this, an ideal value could be w=1.5 and b=0. But any number of other values for w and b justify that equation (because there is only one datapoint).

Both inputs and outputs need to be Tensors. PyTorch tensors are multi-dimensional arrays with the related functionality, plus an optimization to run on CPUs, GPUs, and other devices (CUDA, MPS, CPU, etc.). When requires_grad is set to true, the gradients calculated in loss.backward() are applied by the optimizer. The flag requires_grad is not set to True for x and y, because these values don't change. It (requires_grad) is set to True for w and b (automatically by PyTorch during the model training).

Typical tensors in PyTorch use a datatype of `float32`.

Shape of input tensor x is [1, 1]. This means, there is 1 sample (the first 1) and one item per sample (second 1, which is input to neurons). Same for the output tensor here: number of samples will be the same between input and output, but the output item will be 1 per sample.

In [31]:
x = torch.tensor([[2]], dtype=torch.float32)
y = torch.tensor([[3]], dtype=torch.float32)

print(f"input: {x} output: {y}")
print(f"shape of input: {x.shape} shape of output: {y.shape}")

input: tensor([[2.]]) output: tensor([[3.]])
shape of input: torch.Size([1, 1]) shape of output: torch.Size([1, 1])


## 7. Training Loop

The training loop (one epoch / one batch) follows the following steps:

- set the gradients to zero
- make inference with current weights
- calculate loss
- calculate gradients
- apply gradients
- do the next epoch

### 7.1 Set gradients to 0

`optimizer.zero_grad()`

At the beginning (right after model creation), the gradients would be None. However, gradients will have been calculated after an epoch/loop (when backpropagation is performed). We need to zero-out the gradients for the next loop/epoch; otherwise these gradients will be compounded with new values from the new loop. We want to calculate only what the current parameters have contributed to the loss. So, zero out the gradients.

### 7.2 Make inference with current weights

`y_pred = model(x)`

This model runs through all the samples and provides predictions. Here, there is only one sample, hence there will be only one prediction. Otherwise the y_pred tensor will have multiple predictions (one for each of the samples). The model uses the weight(w) and bias(b) (random ones at the beginning and adjusted ones after the first loop) and calculates y_pred = wx + b. Here the calculation is done only once because there is only one layer / one neuron. With more layers, the inputs will be converted to outputs and fed as inputs to the next layer and so on. The output of the final layer will be y_pred (one value in that tensor).

### 7.3 Calculate loss

`loss = loss_function(y_pred, y)`

Here loss is calculated from the prediction tensor and the original labels (the correct values). In this example, both tensors will have just one value each. Here the error function uses MSE, so, the actual erros if ((y_pred - y) ** 2) / 1. Here, the number of samples is just 1, so, mean is the same as squared error.

### 7.4 Calculate gradients

`loss.backward()`

In the forward propagation model sent the input through layers (above). Now we need to do a back propagation. The `backward()` function calculates the gradient going backward, from output layer to input layer. For each w and b, there will now be gradient values.

### 7.5 Apply Gradients

`optimizer.step()`

Here the optimizer takes the gradient at a parameter and the learning parameter and applies some algorithm (in this case SGD) and changes the values in parameter. 

### 7.6 Do the next epoch

After the optimizer step, the parameters still have gradients. They will be set to None at the beginning of next epoch (loop). And the whole process described above is redone.

In [32]:
def train_model (model, loss_function, optimizer, epochs=range(1), verbose=True):
    # Training loop
    for epoch in epochs:
        # Zero-out gradients
        optimizer.zero_grad()
        print_parameter_info(model, "After setting gradients to None")

        # Inference with current weights
        y_pred = model(x)
        print_parameter_info(model, "After inference")

        # Loss and back propagation
        loss = loss_function(y_pred, y)
        loss.backward()
        print_parameter_info(model, "After backpropagation")

        # Adjust parameters
        optimizer.step()
        print_parameter_info(model, "After Optimizer Step")

## 8. One Epoch of Training

Here one epoch (loop) of training is performed. After each step in the loop parameter info is printed. Analysis below on how the weight and bias are adjusted.

**Zero Grad**

```
After setting gradients to None:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None
```

After `optimizer.zero_grad()`, the gradients are set to None. These values are reset at the beginning of the training loop. With this only the new gradients calculated in the new loop will be appled to params.

**Inference**

```
After inference:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None
```

Inference (using model) will not change any weights, model just uses the existing weights to come up with the inference. No gradient calculation happens either.

**Backpropagation**

```
After backpropagation:
param: tensor([[0.7645]]), requires_grad: True, gradient: tensor([[-2.5637]])
param: tensor([0.8300]), requires_grad: True, gradient: tensor([-1.2818])
```

After backpropagation, you can see the gradients calculated. However, these are not applied to the values of weights yet.

**Optimizer**

```
After Optimizer Step:
param: tensor([[1.0209]]), requires_grad: True, gradient: tensor([[-2.5637]])
param: tensor([0.9582]), requires_grad: True, gradient: tensor([-1.2818])
```

The `optimizer.step()` adjust the values of the weights. However, the gradients have not yet been cleared. This is done at the beginning of next loop (epoch).

In [33]:
train_model(model, loss_function, optimizer)


After setting gradients to None:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None

After inference:
param: tensor([[0.7645]]), requires_grad: True, gradient: None
param: tensor([0.8300]), requires_grad: True, gradient: None

After backpropagation:
param: tensor([[0.7645]]), requires_grad: True, gradient: tensor([[-2.5637]])
param: tensor([0.8300]), requires_grad: True, gradient: tensor([-1.2818])

After Optimizer Step:
param: tensor([[1.0209]]), requires_grad: True, gradient: tensor([[-2.5637]])
param: tensor([0.9582]), requires_grad: True, gradient: tensor([-1.2818])
